In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_feasibility.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/run_lhs.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_design_space.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_distance.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_predict.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_sampler.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_uncertainty.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_config.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_explain.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/test_lhs.py
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation/lhs_input_schema_audit.csv
/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-

In [4]:
import os
import shutil

INPUT_DIR = "/kaggle/input/datasets/sakshamsrivastava26/encode-run"
DATA_PATH = "/kaggle/input/datasets/sakshamsrivastava26/perla-final-filtered-2/perla_final_filtered.pkl"
WORK_DIR = "/kaggle/working/psc_project"

os.makedirs(WORK_DIR, exist_ok=True)

# Check whether the required encoder exists anywhere in Kaggle inputs.
encoder_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "perla_encoding_by_claude.py" in files:
        encoder_path = os.path.join(root, "perla_encoding_by_claude.py")
        break

print("Encoder found:", encoder_path)

# Copy the project scripts and dataset into the writable directory.
for filename in os.listdir(INPUT_DIR):
    if filename.endswith(".py"):
        shutil.copy2(
            os.path.join(INPUT_DIR, filename),
            os.path.join(WORK_DIR, filename)
        )

shutil.copy2(DATA_PATH, os.path.join(WORK_DIR, "perla_final_filtered.pkl"))

if encoder_path:
    shutil.copy2(
        encoder_path,
        os.path.join(WORK_DIR, "perla_encoding_by_claude.py")
    )

print("Project prepared at:", WORK_DIR)
print("Files:", sorted(os.listdir(WORK_DIR)))

Encoder found: /kaggle/input/datasets/sakshamsrivastava26/encode-run/perla_encoding_by_claude.py
Project prepared at: /kaggle/working/psc_project
Files: ['perla_cv.py', 'perla_encoding_by_claude.py', 'perla_explain_uq.py', 'perla_final_filtered.pkl', 'perla_mapie.py', 'perla_shap.py', 'run_models.py', 'test_candidates.py', 'test_explain_uq.py', 'test_split_pipeline.py']


In [5]:
!pip install matminer
!pip install mapie xgboost

import os
import sys
import shutil

WORK_DIR = "/kaggle/working/psc_project"
os.chdir(WORK_DIR)

# Ensure the working directory is importable.
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import perla_cv

RUN_MODELS_PATH = os.path.join(WORK_DIR, "run_models.py")

with open(RUN_MODELS_PATH, "r", encoding="utf-8") as f:
    content = f.read()

injection = (
    "\nimport sys\n"
    "import perla_cv\n"
    "sys.modules['__main__'].FoldData = perla_cv.FoldData\n"
)

marker = "sys.modules['__main__'].FoldData"

if marker not in content:
    if "import itertools" not in content:
        raise RuntimeError(
            "Could not find the expected 'import itertools' insertion point. "
            "Inspect the imports in run_models.py before patching."
        )

    content = content.replace(
        "import itertools",
        "import itertools" + injection,
        1
    )

    with open(RUN_MODELS_PATH, "w", encoding="utf-8") as f:
        f.write(content)

    print("Patched run_models.py for FoldData unpickling.")
else:
    print("FoldData unpickling patch already exists.")

print("Script:", RUN_MODELS_PATH)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.3 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.3 MB/s eta 0:00:00ta 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 48.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 1.2 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.8/825.8 kB 16.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 48.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 37.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 6.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 18.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 1.3 MB/s eta 0:00:00eta 0:00:01
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=436

In [6]:
%cd /kaggle/working/psc_project
!python perla_cv.py --data perla_final_filtered.pkl --folds 5 --cache-dir cv_cache --seed 42

/kaggle/working
Total rows            : 39,151
Train rows            : 27,410  (70.01%)   unique DOI groups: 5,910
Validation rows       : 5,870  (14.99%)   unique DOI groups: 1,302
Test rows             : 5,871  (15.00%)   unique DOI groups: 1,223
Train/Validation DOI overlap: 0
Train/Test DOI overlap: 0
Validation/Test DOI overlap: 0
Note: the split is by whole papers, so the row percentages deviate slightly from 70/15/15 (papers contribute different numbers of rows).
5,910 papers in TRAIN, 5 GroupKFold folds
  fold 0: train (21928, 293)  val (5482, 293)  papers train/val 4729/1181  [fit+transform train 48s, transform val 13s]
  fold 1: train (21928, 289)  val (5482, 289)  papers train/val 4729/1181  [fit+transform train 47s, transform val 11s]
  fold 2: train (21928, 293)  val (5482, 293)  papers train/val 4728/1182  [fit+transform train 46s, transform val 11s]
  fold 3: train (21928, 291)  val (5482, 291)  papers train/val 4727/1183  [fit+transform train 46s, transform val 11s]
  f

In [7]:
!python run_models.py --data perla_final_filtered.pkl --cache-dir cv_cache --out results --folds 5 --inner-folds 5 --seed 0 --split-seed 42 --n-jobs -1

Total rows            : 39,151
Train rows            : 27,410  (70.01%)   unique DOI groups: 5,910
Validation rows       : 5,870  (14.99%)   unique DOI groups: 1,302
Test rows             : 5,871  (15.00%)   unique DOI groups: 1,223
Train/Validation DOI overlap: 0
Train/Test DOI overlap: 0
Validation/Test DOI overlap: 0
Note: the split is by whole papers, so the row percentages deviate slightly from 70/15/15 (papers contribute different numbers of rows).
run id: 5fold_seed0_split42_60e10f71  ->  results/5fold_seed0_split42_60e10f71
CV on the 27,410 TRAIN rows, 5,910 papers
  fold 0: loaded from cache
  fold 0: 12 candidates scored; lowest RMSE this fold 3.341 (TwoStage[S1=HistGradientBoosting]|VotingET); stage1 159s
  fold 1: loaded from cache
  fold 1: 12 candidates scored; lowest RMSE this fold 3.246 (TwoStage[S1=HistGradientBoosting]|VotingET); stage1 154s
  fold 2: loaded from cache
  fold 2: 12 candidates scored; lowest RMSE this fold 3.312 (TwoStage[S1=HistGradientBoosting]|Votin

In [11]:
%cd /kaggle/working/psc_project

!python -u perla_explain_uq.py \
    --data /kaggle/working/psc_project/perla_final_filtered.pkl \
    --out /kaggle/working/psc_project/results_uq \
    --mapie-models VotingET TwoStage:VotingET \
    --cal-repeats 5

/kaggle/working
run id: uq_seed0_split42_47f13251 -> /kaggle/working/psc_project/results_uq/uq_seed0_split42_47f13251
requested: MAPIE ['Direct|VotingET', 'TwoStage[S1=HistGradientBoosting]|VotingET'] | SHAP no | test set not used
Total rows            : 39,151
Train rows            : 27,410  (70.01%)   unique DOI groups: 5,910
Validation rows       : 5,870  (14.99%)   unique DOI groups: 1,302
Test rows             : 5,871  (15.00%)   unique DOI groups: 1,223
Train/Validation DOI overlap: 0
Train/Test DOI overlap: 0
Validation/Test DOI overlap: 0
Note: the split is by whole papers, so the row percentages deviate slightly from 70/15/15 (papers contribute different numbers of rows).
preprocessing restored from cache prep_4f3cba6b74f3.joblib (311 features)

=== MAPIE CQR for 2 requested candidate(s): ['Direct|VotingET', 'TwoStage[S1=HistGradientBoosting]|VotingET'] ===

--- Direct|VotingET ---
  [Direct] RandomForest: point model fitted in 31s
  [Direct] XGBoost: point model fitted in 6s


In [12]:
!python /kaggle/working/psc_project/perla_explain_uq.py \
    --shap-models VotingET TwoStage:VotingET

run id: uq_seed0_split42_e04bab08 -> results_uq/uq_seed0_split42_e04bab08
requested: MAPIE no | SHAP ['Direct|VotingET', 'TwoStage[S1=HistGradientBoosting]|VotingET'] | test set not used
Total rows            : 39,151
Train rows            : 27,410  (70.01%)   unique DOI groups: 5,910
Validation rows       : 5,870  (14.99%)   unique DOI groups: 1,302
Test rows             : 5,871  (15.00%)   unique DOI groups: 1,223
Train/Validation DOI overlap: 0
Train/Test DOI overlap: 0
Validation/Test DOI overlap: 0
Note: the split is by whole papers, so the row percentages deviate slightly from 70/15/15 (papers contribute different numbers of rows).
preprocessing restored from cache prep_4f3cba6b74f3.joblib (311 features)

=== SHAP (interventional) for 2 requested candidate(s): ['Direct|VotingET', 'TwoStage[S1=HistGradientBoosting]|VotingET'] ===

--- Direct|VotingET ---
  [Direct] RandomForest: point model fitted in 33s
100%|===================| 999/1000 [14:45<00:00]          [Direct] RandomFore

In [13]:
import shutil
from pathlib import Path

project_dir = Path("/kaggle/working/psc_project")
zip_path = "/kaggle/working/psc_project"

if not project_dir.exists():
    raise FileNotFoundError(f"Folder not found: {project_dir}")

archive = shutil.make_archive(
    zip_path,
    "zip",
    root_dir="/kaggle/working",
    base_dir="psc_project"
)

print("Created:", archive)
print("Size (MB):", round(Path(archive).stat().st_size / (1024**2), 2))

Created: /kaggle/working/psc_project.zip
Size (MB): 32.58


In [14]:
import os
import shutil

LHS_INPUT = "/kaggle/input/datasets/sakshamsrivastava26/candidate-grid-generation"
WORK_DIR = "/kaggle/working/lhs_project"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "outputs"), exist_ok=True)

# Copy all LHS Python scripts and supporting files.
for filename in os.listdir(LHS_INPUT):
    if filename.endswith((".py", ".csv", ".md")):
        shutil.copy2(
            os.path.join(LHS_INPUT, filename),
            os.path.join(WORK_DIR, filename)
        )

# Copy the model pipeline scripts and encoder.
MODEL_INPUT = "/kaggle/input/datasets/sakshamsrivastava26/encode-run"

for filename in os.listdir(MODEL_INPUT):
    if filename.endswith(".py"):
        shutil.copy2(
            os.path.join(MODEL_INPUT, filename),
            os.path.join(WORK_DIR, filename)
        )

# Copy the dataset.
DATA_PATH = "/kaggle/input/datasets/sakshamsrivastava26/perla-final-filtered-2/perla_final_filtered.pkl"

shutil.copy2(
    DATA_PATH,
    os.path.join(WORK_DIR, "perla_final_filtered.pkl")
)

print("LHS script exists:", os.path.isfile(os.path.join(WORK_DIR, "run_lhs.py")))
print("Project files:", sorted(os.listdir(WORK_DIR)))

%cd /kaggle/working/lhs_project

!pip install numpy pandas scipy scikit-learn xgboost mapie shap matminer pymatgen joblib pytest

LHS script exists: True
Project files: ['README_LHS.md', 'lhs_config.py', 'lhs_design_space.py', 'lhs_distance.py', 'lhs_explain.py', 'lhs_feasibility.py', 'lhs_feature_bounds.csv', 'lhs_input_schema_audit.csv', 'lhs_predict.py', 'lhs_rank.py', 'lhs_sampler.py', 'lhs_uncertainty.py', 'outputs', 'perla_cv.py', 'perla_encoding_by_claude.py', 'perla_explain_uq.py', 'perla_final_filtered.pkl', 'perla_mapie.py', 'perla_shap.py', 'run_lhs.py', 'run_models.py', 'test_candidates.py', 'test_explain_uq.py', 'test_lhs.py', 'test_split_pipeline.py']
/kaggle/working/lhs_project


In [15]:
!python run_lhs.py \
  --data perla_final_filtered.pkl \
  --out /kaggle/working/lhs_project/outputs \
  --reference-uq-dir results_uq/uq_seed0_split42_47f13251 \
--reference-prep-cache results_uq/_prep_cache/prep_4f3cba6b74f3.joblib

design space from 27410 TRAIN rows / 5910 papers: 14 families, 54/17/13 ETL/HTL/BC recipes, 15 process recipes
sampled 50000 raw candidates; exact LHS strata in all dimensions: True
funnel: {'stage': 'raw LHS candidates', 'remaining': 50000, 'removed': 0}
funnel: {'stage': 'after invalid-candidate rules', 'remaining': 49929, 'removed': 71}
funnel: {'stage': 'after fabrication-incompatibility rules', 'remaining': 47746, 'removed': 2183}
funnel: {'stage': 'feasible (flagged uncommon among them: 36611)', 'remaining': 47746, 'removed': 0}
[artifacts] split: train 27410 / val 5870 / test 5871 rows
[artifacts] preprocessors fitted on TRAIN in 80s: 311 features
[artifacts] cache comparison: {'available': False, 'note': 'no cached training matrix given (cfg.reference_prep_cache); reproduction of the reported run not verified'}
[encode] 5000/47746
[encode] 10000/47746
[encode] 15000/47746
[encode] 20000/47746
[encode] 25000/47746
[encode] 30000/47746
[encode] 35000/47746
[encode] 40000/47746
[e

In [16]:
import shutil
from pathlib import Path

project_dir = Path("/kaggle/working/lhs_project")
zip_path = "/kaggle/working/lhs_project"

if not project_dir.exists():
    raise FileNotFoundError(f"Folder not found: {project_dir}")

archive = shutil.make_archive(
    zip_path,
    "zip",
    root_dir="/kaggle/working",
    base_dir="lhs_project"
)

print("Created:", archive)
print("Size (MB):", round(Path(archive).stat().st_size / (1024**2), 2))

Created: /kaggle/working/lhs_project.zip
Size (MB): 26.87
